#### **Figures 2–4: Genome-wide coverage — Native RNA vs IVT RNA (*E. coli* K-12)**

Plots per-base sequencing coverage (log₁₀(CPM + 1)) across the *E. coli* K-12 genome for Native RNA and IVT RNA samples, with rRNA regions highlighted in red. Set the data path in the cell below before running.

**Samples:** Native RNA (3 replicates), IVT RNA (3 replicates)

**Reference:** *E. coli* K-12 (ENA|U00096|U00096.3)

**rRNA regions:** 7 operons (16S, 23S, 5S) — coordinates in `rrna_coordinates.csv`

#### **Figures in this notebook: Genome-wide Sequencing Coverage — Native RNA vs IVT RNA**

| Figure | Description |
|--------|-------------|
| **Fig. 2** | Genome-wide coverage scatter — native and IVT (2-panel) |
| **Fig. 3** | Genome-wide coverage scatter, native RNA — 5 × 1 Mbp panels |
| **Fig. 4** | Genome-wide coverage scatter, IVT RNA — 5 × 1 Mbp panels |

Binned variants (100 bp and 1 kb, scatter/line/step) are also generated for each figure.

---

### Requirements

- **Python** 3.10+ with `matplotlib`, `seaborn`, `pandas`, `numpy`
- **Read counts**: obtained with `samtools view -c -F 2308` (primary alignments only) — hardcoded in the read counts cell

### Input file layout

Place the notebook and the following files/folders in the same directory:

```
your_folder/
├── figs-2-3-4.ipynb
├── rrna_coordinates.csv
└── coverages/
    ├── native/
    │   ├── k12_native_full_bc1.txt
    │   ├── k12_native_full_bc2.txt
    │   └── k12_native_full_bc3.txt
    └── ivt/
        ├── k12_ivt_full_bc1.txt
        ├── k12_ivt_full_bc2.txt
        └── k12_ivt_full_bc3.txt
```

Coverage files are tab-separated with columns `chrom`, `position`, `depth` (standard `samtools depth` output). Output figures are saved to `output/figs-pngs/` and `output/figs-svgs/` in the same folder in 600 dpi quality

In [9]:
## import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 8)


> **Note:** `DATA_DIR` resolves to the directory Jupyter was launched from, which is normally the notebook's folder. If figures fail to load, set `DATA_DIR` explicitly to an absolute path.

In [10]:
# set to the directory containing coverages/ and rrna_coordinates.csv
DATA_DIR = Path('.').resolve()
# DATA_DIR = Path('/absolute/path/to/your/data')

In [11]:
# Load rRNA coordinates
rrna_coords = pd.read_csv(DATA_DIR / 'rrna_coordinates.csv')
print("rRNA regions in E. coli K12:")
print(rrna_coords)

rRNA regions in E. coli K12:
    gene    start      end
0   rrsH   223771   225312
1   rrlH   225759   228662
2   rrfH   228756   228875
3   rrfG  2726069  2726188
4   rrlG  2726281  2729184
5   rrsG  2729616  2731157
6   rrfF  3423423  3423542
7   rrfD  3423668  3423787
8   rrlD  3423880  3426783
9   rrsD  3427221  3428762
10  rrsC  3941808  3943349
11  rrlC  3943704  3946607
12  rrfC  3946700  3946819
13  rrsA  4035531  4037072
14  rrlA  4037519  4040423
15  rrfA  4040517  4040636
16  rrsB  4166659  4168200
17  rrlB  4168641  4171544
18  rrfB  4171637  4171756
19  rrsE  4208147  4209688
20  rrlE  4210043  4212946
21  rrfE  4213040  4213159


In [12]:
## Coverage file paths

native_files = [
    DATA_DIR / 'coverages/native/k12_native_full_bc1.txt',
    DATA_DIR / 'coverages/native/k12_native_full_bc2.txt',
    DATA_DIR / 'coverages/native/k12_native_full_bc3.txt',
]

ivt_files = [
    DATA_DIR / 'coverages/ivt/k12_ivt_full_bc1.txt',
    DATA_DIR / 'coverages/ivt/k12_ivt_full_bc2.txt',
    DATA_DIR / 'coverages/ivt/k12_ivt_full_bc3.txt',
]

In [13]:
# Function to load coverage data
def load_coverage_data(file_path):
    df = pd.read_csv(file_path, sep='	', header=None, names=['chrom', 'position', 'coverage'])
    return df

# Load all datasets
native_reps = [load_coverage_data(f) for f in native_files]
ivt_reps = [load_coverage_data(f) for f in ivt_files]

# df shape check - sanity check
print(f"Native replicate shapes: {native_reps[0].shape}, {native_reps[1].shape}, {native_reps[2].shape}")
print(f"IVT replicate shapes: {ivt_reps[0].shape}, {ivt_reps[1].shape}, {ivt_reps[2].shape}")


Native replicate shapes: (4641652, 3), (4641652, 3), (4641652, 3)
IVT replicate shapes: (4641652, 3), (4641652, 3), (4641652, 3)


In [14]:
# read counts from samtools view -c -F 2308
read_counts = {
    'native_rep1': 88366,
    'native_rep2': 235984,
    'native_rep3': 347917,
    'ivt_rep1': 72626,
    'ivt_rep2': 50142,
    'ivt_rep3': 49260,
}

for i in range(1, 4):
    print(f"Native replicate {i}: {read_counts[f'native_rep{i}']:,} reads")
for i in range(1, 4):
    print(f"IVT replicate {i}: {read_counts[f'ivt_rep{i}']:,} reads")

Native replicate 1: 88,366 reads
Native replicate 2: 235,984 reads
Native replicate 3: 347,917 reads
IVT replicate 1: 72,626 reads
IVT replicate 2: 50,142 reads
IVT replicate 3: 49,260 reads


In [15]:
df_plot = native_reps[0][['chrom', 'position']].copy()

# total reads per condition (pooled across replicates)
native_total_reads = read_counts['native_rep1'] + read_counts['native_rep2'] + read_counts['native_rep3']
ivt_total_reads = read_counts['ivt_rep1'] + read_counts['ivt_rep2'] + read_counts['ivt_rep3']

# sum coverage across replicates
native_pooled_coverage = native_reps[0]['coverage'] + native_reps[1]['coverage'] + native_reps[2]['coverage']
ivt_pooled_coverage = ivt_reps[0]['coverage'] + ivt_reps[1]['coverage'] + ivt_reps[2]['coverage']

df_plot['native_cpm'] = (native_pooled_coverage / native_total_reads) * 1_000_000
df_plot['ivt_cpm'] = (ivt_pooled_coverage / ivt_total_reads) * 1_000_000

df_plot['native_log_cpm'] = np.log10(df_plot['native_cpm'] + 1)
df_plot['ivt_log_cpm'] = np.log10(df_plot['ivt_cpm'] + 1)

print(df_plot[['native_cpm', 'ivt_cpm', 'native_log_cpm', 'ivt_log_cpm']].describe())

         native_cpm       ivt_cpm  native_log_cpm   ivt_log_cpm
count  4.641652e+06  4.641652e+06    4.641652e+06  4.641652e+06
mean   1.461400e+02  9.219987e+01    5.935455e-01  3.051341e-01
std    2.719319e+03  1.910813e+03    6.449636e-01  5.713141e-01
min    0.000000e+00  0.000000e+00    0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00    0.000000e+00  0.000000e+00
50%    1.487504e+00  0.000000e+00    3.957638e-01  0.000000e+00
75%    7.437521e+00  5.813007e+00    9.262149e-01  8.333388e-01
max    1.133701e+05  9.549027e+04    5.054502e+00  4.979964e+00


In [16]:
# Create a column to mark rRNA regions
df_plot['is_rrna'] = False

for _, row in rrna_coords.iterrows():
    mask = (df_plot['position'] >= row['start']) & (df_plot['position'] <= row['end'])
    df_plot.loc[mask, 'is_rrna'] = True

print(f"Positions in rRNA regions: {df_plot['is_rrna'].sum():,}")
print(f"Positions in non-rRNA regions: {(~df_plot['is_rrna']).sum():,}")


Positions in rRNA regions: 32,083
Positions in non-rRNA regions: 4,609,569


### Output folders and plotting helpers


In [17]:
# output folders
PNG_ROOT = Path('output/figs-pngs')
SVG_ROOT = Path('output/figs-svgs')


def save_current_figure(rel_dir, base_name):
    for root, ext, kwargs in [
        (PNG_ROOT, 'png', {'dpi': 600}),
        (SVG_ROOT, 'svg', {})
    ]:
        out_dir = root / rel_dir
        out_dir.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_dir / f"{base_name}.{ext}", bbox_inches='tight', **kwargs)


# plot styling presets
RC_GENOMEWIDE = {
    'font.size': 16,
    'axes.labelsize': 20,
    'axes.titlesize': 22,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 16,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial'],
    'axes.linewidth': 2,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'xtick.major.size': 7,
    'ytick.major.size': 7,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
}

RC_MULTIPANEL = {
    'font.size': 14,
    'axes.labelsize': 18,
    'axes.titlesize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 14,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial'],
    'axes.linewidth': 2,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
}

# colors
non_rrna_color = '#6B7280'  # gray
rrna_color = '#DC2626'      # red

# genome windows for multi-panel plots
windows = [
    (0, 1_000_000, '0-1'),
    (1_000_000, 2_000_000, '1-2'),
    (2_000_000, 3_000_000, '2-3'),
    (3_000_000, 4_000_000, '3-4'),
    (4_000_000, 4_641_652, '4-4.64')
]

# style parameters by bin size
STYLE_CONFIG = {
    'pooled': {
        'scatter': {
            'non': {'s': 2, 'alpha': 0.5},
            'rrna': {'s': 2, 'alpha': 0.7}
        }
    },
    '100bp': {
        'scatter': {
            'non': {'s': 3, 'alpha': 0.6},
            'rrna': {'s': 3, 'alpha': 0.7}
        },
        'line': {
            'non': {'linewidth': 0.6, 'alpha': 0.9},
            'rrna': {'linewidth': 1.0, 'alpha': 0.95}
        },
        'step': {
            'non': {'linewidth': 0.8, 'alpha': 0.9},
            'rrna': {'linewidth': 1.2, 'alpha': 0.95}
        }
    },
    '1kb': {
        'scatter': {
            'non': {'s': 4, 'alpha': 0.7},
            'rrna': {'s': 4, 'alpha': 0.8}
        },
        'line': {
            'non': {'linewidth': 0.9, 'alpha': 0.9},
            'rrna': {'linewidth': 1.3, 'alpha': 0.95}
        },
        'step': {
            'non': {'linewidth': 1.0, 'alpha': 0.9},
            'rrna': {'linewidth': 1.4, 'alpha': 0.95}
        }
    }
}


def plot_series(ax, x, y, style, color, label, params):
    if style == 'scatter':
        ax.scatter(x, y, c=color, label=label, rasterized=True, **params)
    elif style == 'line':
        ax.plot(x, y, color=color, label=label, **params)
    elif style == 'step':
        ax.step(x, y, where='mid', color=color, label=label, **params)
    else:
        raise ValueError(f"Unknown style: {style}")


def plot_genomewide(df, style, cfg, out_dir, base_name):
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    ax1 = axes[0]
    if style == 'scatter':
        non = df[~df['is_rrna']]
        rrna = df[df['is_rrna']]
        plot_series(ax1, non['position'], non['native_log_cpm'], style, non_rrna_color, 'Non-rRNA', cfg['non'])
        plot_series(ax1, rrna['position'], rrna['native_log_cpm'], style, rrna_color, 'rRNA', cfg['rrna'])
    else:
        plot_series(ax1, df['position'], df['native_log_cpm'], style, non_rrna_color, 'Non-rRNA', cfg['non'])
        rrna = df['native_log_cpm'].where(df['is_rrna'])
        plot_series(ax1, df['position'], rrna, style, rrna_color, 'rRNA', cfg['rrna'])

    ax1.set_ylabel('log10(CPM + 1)', fontsize=22, fontweight='bold')
    ax1.legend(loc='upper right', bbox_to_anchor=(1.05, 1.0), frameon=True, fancybox=False,
               shadow=False, framealpha=0.95, edgecolor='black',
               markerscale=5, fontsize=18)
    ax1.grid(True, alpha=0.2, linewidth=1)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    ax2 = axes[1]
    if style == 'scatter':
        non = df[~df['is_rrna']]
        rrna = df[df['is_rrna']]
        plot_series(ax2, non['position'], non['ivt_log_cpm'], style, non_rrna_color, 'Non-rRNA', cfg['non'])
        plot_series(ax2, rrna['position'], rrna['ivt_log_cpm'], style, rrna_color, 'rRNA', cfg['rrna'])
    else:
        plot_series(ax2, df['position'], df['ivt_log_cpm'], style, non_rrna_color, 'Non-rRNA', cfg['non'])
        rrna = df['ivt_log_cpm'].where(df['is_rrna'])
        plot_series(ax2, df['position'], rrna, style, rrna_color, 'rRNA', cfg['rrna'])

    ax2.set_xlabel('Genome Position (Mbp)', fontsize=22, fontweight='bold')
    ax2.set_ylabel('log10(CPM + 1)', fontsize=22, fontweight='bold')
    ax2.legend(loc='upper right', bbox_to_anchor=(1.05, 1.0), frameon=True, fancybox=False,
               shadow=False, framealpha=0.95, edgecolor='black',
               markerscale=5, fontsize=18)
    ax2.grid(True, alpha=0.2, linewidth=1)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    ax2.ticklabel_format(style='plain', axis='x')
    ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}' if x >= 1e6 else f'{x/1e3:.0f}'))

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.15)
    save_current_figure(out_dir, base_name)
    plt.close(fig)


def plot_multipanel(df, style, cfg, value_col, out_dir, base_name):
    fig, axes = plt.subplots(5, 1, figsize=(16, 20), sharex=False, sharey=True)

    for idx, (start, end, label) in enumerate(windows):
        ax = axes[idx]
        mask = (df['position'] >= start) & (df['position'] < end)
        window_data = df[mask].copy()

        if style == 'scatter':
            non = window_data[~window_data['is_rrna']]
            rrna = window_data[window_data['is_rrna']]
            plot_series(ax, non['position'], non[value_col], style, non_rrna_color,
                        'Non-rRNA' if idx == 0 else '', cfg['non'])
            plot_series(ax, rrna['position'], rrna[value_col], style, rrna_color,
                        'rRNA' if idx == 0 else '', cfg['rrna'])
        else:
            plot_series(ax, window_data['position'], window_data[value_col], style, non_rrna_color,
                        'Non-rRNA' if idx == 0 else '', cfg['non'])
            rrna = window_data[value_col].where(window_data['is_rrna'])
            plot_series(ax, window_data['position'], rrna, style, rrna_color,
                        'rRNA' if idx == 0 else '', cfg['rrna'])

        ax.set_xlim(start, start + 1_000_000)
        if idx == 2:
            ax.set_ylabel('log10(CPM + 1)', fontsize=20, fontweight='bold')
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}'))
        ax.text(1.02, 0.5, f'{label} Mbp', transform=ax.transAxes,
                fontsize=16, va='center', ha='left')
        ax.grid(True, alpha=0.2, linewidth=0.8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if idx == 0:
            ax.legend(loc='upper right', frameon=True, fancybox=False,
                      shadow=False, framealpha=0.95, edgecolor='black',
                      markerscale=4, fontsize=16)

    axes[-1].set_xlabel('Genome Position (Mbp)', fontsize=20, fontweight='bold')
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.25, right=0.92)
    save_current_figure(out_dir, base_name)
    plt.close(fig)

### Figures 2-4: Pooled CPM (scatter)


In [18]:
# Pooled CPM (scatter only)
plt.rcParams.update(RC_GENOMEWIDE)
plot_genomewide(
    df_plot,
    style='scatter',
    cfg=STYLE_CONFIG['pooled']['scatter'],
    out_dir=Path('pooled_CPM'),
    base_name='fig2_pooled_scatter'
)

plt.rcParams.update(RC_MULTIPANEL)
plot_multipanel(
    df_plot,
    style='scatter',
    cfg=STYLE_CONFIG['pooled']['scatter'],
    value_col='native_log_cpm',
    out_dir=Path('pooled_CPM'),
    base_name='fig3_pooled_scatter_native'
)
plot_multipanel(
    df_plot,
    style='scatter',
    cfg=STYLE_CONFIG['pooled']['scatter'],
    value_col='ivt_log_cpm',
    out_dir=Path('pooled_CPM'),
    base_name='fig4_pooled_scatter_ivt'
)

print('Saved pooled CPM scatter plots (fig2/fig3/fig4).')


Saved pooled CPM scatter plots (fig2/fig3/fig4).


### Binning (100 bp and 1 kb)


In [19]:
def bin_cpm(df, bin_size):
    binned = df.copy()
    binned['bin_start'] = ((binned['position'] - 1) // bin_size) * bin_size + 1
    agg = binned.groupby('bin_start', sort=True).agg(
        native_cpm=('native_cpm', 'mean'),
        ivt_cpm=('ivt_cpm', 'mean'),
        is_rrna=('is_rrna', 'any')
    ).reset_index()
    agg['position'] = agg['bin_start'] + (bin_size - 1) / 2.0
    agg['native_log_cpm'] = np.log10(agg['native_cpm'] + 1)
    agg['ivt_log_cpm'] = np.log10(agg['ivt_cpm'] + 1)
    return agg

binned_100bp = bin_cpm(df_plot, 100)
binned_1kb = bin_cpm(df_plot, 1000)

print('100 bp bins:', binned_100bp.shape)
print('1 kb bins:', binned_1kb.shape)

100 bp bins: (46417, 7)
1 kb bins: (4642, 7)


### Figures 2-4: Binned CPM (scatter, line, step)


In [20]:
# Generate scatter, line, and step plots for both bin sizes
for bin_label, df_bin in [('100bp', binned_100bp), ('1kb', binned_1kb)]:
    for style in ['scatter', 'line', 'step']:
        cfg = STYLE_CONFIG[bin_label][style]
        rel_dir = Path(f'binned_{bin_label}') / style

        plt.rcParams.update(RC_GENOMEWIDE)
        plot_genomewide(
            df_bin,
            style=style,
            cfg=cfg,
            out_dir=rel_dir,
            base_name=f'fig2_binned{bin_label}_{style}'
        )

        plt.rcParams.update(RC_MULTIPANEL)
        plot_multipanel(
            df_bin,
            style=style,
            cfg=cfg,
            value_col='native_log_cpm',
            out_dir=rel_dir,
            base_name=f'fig3_binned{bin_label}_{style}_native'
        )
        plot_multipanel(
            df_bin,
            style=style,
            cfg=cfg,
            value_col='ivt_log_cpm',
            out_dir=rel_dir,
            base_name=f'fig4_binned{bin_label}_{style}_ivt'
        )

print('Saved all binned plots for 100 bp and 1 kb (scatter, line, step).')


Saved all binned plots for 100 bp and 1 kb (scatter, line, step).
